In [1]:
import pandas as pd
import numpy as np
import os
import json
import pickle

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

# Load clean dataset
df = pd.read_csv('../extracted/ciciot_clean.csv')
print(f"Loaded: {df.shape}")

# -----------
# 1. targets
# -----------

# Binary target
df['target_binary'] = (df['Label'] != 'BENIGN').astype(int)

# Fine-grained 34-class target
fine_le = LabelEncoder()
df['target_fine34'] = fine_le.fit_transform(df['Label'])

fine34_mapping = {label: int(idx) for idx, label in enumerate(fine_le.classes_)}

# 8-category mapping
CATEGORY_ENCODING = {
    'DDoS': 0,
    'DoS': 1,
    'Mirai': 2,
    'Benign': 3,
    'Recon': 4,
    'Spoofing': 5,
    'Web': 6,
    'BruteForce': 7
}

df['target_cat8'] = df['Category'].map(CATEGORY_ENCODING)

if df['target_cat8'].isna().sum() > 0:
    missing_categories = df.loc[df['target_cat8'].isna(), 'Category'].unique()
    raise ValueError(f"Unmapped categories found: {missing_categories}")

df['target_cat8'] = df['target_cat8'].astype(int)

print("Binary counts:")
print(df['target_binary'].value_counts())

print("\nFine34 counts:")
print(df['target_fine34'].value_counts().sort_index().head())

print("\nCat8 counts:")
print(df['target_cat8'].value_counts().sort_index())

Loaded: (21005152, 41)
Binary counts:
target_binary
1    19957844
0     1047308
Name: count, dtype: int64

Fine34 counts:
target_fine34
0       3075
1    1047308
2       5560
3       5150
4     271535
Name: count, dtype: int64

Cat8 counts:
target_cat8
0    12292065
1     4178844
2     2359183
3     1047308
4      655464
5      436061
6       23707
7       12520
Name: count, dtype: int64


In [2]:
# -----------------------------
# 2. Defining feature columns
# -----------------------------
exclude_cols = ['Label', 'Category', 'target_binary', 'target_fine34', 'target_cat8']
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols]

y_binary = df['target_binary']
y_fine34 = df['target_fine34']
y_cat8 = df['target_cat8']

print(f"Number of features: {len(feature_cols)}")
print(feature_cols)

Number of features: 39
['Header_Length', 'Protocol Type', 'Time_To_Live', 'Rate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Variance']


In [3]:
# ------------------
# 3A. Binary split
# ------------------
X_temp_b, X_test_b, y_temp_b, y_test_b = train_test_split(
    X, y_binary,
    test_size=0.20,
    random_state=42,
    stratify=y_binary
)

X_train_b, X_val_b, y_train_b, y_val_b = train_test_split(
    X_temp_b, y_temp_b,
    test_size=0.125,   # 10% of total
    random_state=42,
    stratify=y_temp_b
)

print("Binary split complete")
print(f"Train: {X_train_b.shape}, Val: {X_val_b.shape}, Test: {X_test_b.shape}")
print("Train binary distribution:", y_train_b.value_counts().to_dict())
print("Val binary distribution:", y_val_b.value_counts().to_dict())
print("Test binary distribution:", y_test_b.value_counts().to_dict())

Binary split complete
Train: (14703605, 39), Val: (2100516, 39), Test: (4201031, 39)
Train binary distribution: {1: 13970490, 0: 733115}
Val binary distribution: {1: 1995785, 0: 104731}
Test binary distribution: {1: 3991569, 0: 209462}


In [4]:
# ----------------------
# 3B. 8-category split
# ----------------------
X_temp_m, X_test_m, y_temp_cat8, y_test_cat8, y_temp_fine34, y_test_fine34 = train_test_split(
    X, y_cat8, y_fine34,
    test_size=0.20,
    random_state=42,
    stratify=y_cat8
)

X_train_m, X_val_m, y_train_cat8, y_val_cat8, y_train_fine34, y_val_fine34 = train_test_split(
    X_temp_m, y_temp_cat8, y_temp_fine34,
    test_size=0.125,
    random_state=42,
    stratify=y_temp_cat8
)

print("Cat8 split complete")
print(f"Train: {X_train_m.shape}, Val: {X_val_m.shape}, Test: {X_test_m.shape}")
print("Train cat8 distribution:", y_train_cat8.value_counts().sort_index().to_dict())
print("Val cat8 distribution:", y_val_cat8.value_counts().sort_index().to_dict())
print("Test cat8 distribution:", y_test_cat8.value_counts().sort_index().to_dict())

Cat8 split complete
Train: (14703605, 39), Val: (2100516, 39), Test: (4201031, 39)
Train cat8 distribution: {0: 8604445, 1: 2925190, 2: 1651428, 3: 733115, 4: 458825, 5: 305243, 6: 16595, 7: 8764}
Val cat8 distribution: {0: 1229207, 1: 417885, 2: 235918, 3: 104731, 4: 65546, 5: 43606, 6: 2371, 7: 1252}
Test cat8 distribution: {0: 2458413, 1: 835769, 2: 471837, 3: 209462, 4: 131093, 5: 87212, 6: 4741, 7: 2504}


In [5]:
# 4A. Scale binary branch

scaler_binary = MinMaxScaler()

X_train_b_scaled = pd.DataFrame(
    scaler_binary.fit_transform(X_train_b),
    columns=feature_cols
)
X_val_b_scaled = pd.DataFrame(
    scaler_binary.transform(X_val_b),
    columns=feature_cols
)
X_test_b_scaled = pd.DataFrame(
    scaler_binary.transform(X_test_b),
    columns=feature_cols
)

# 4B. Scale multiclass branch

scaler_cat8 = MinMaxScaler()

X_train_m_scaled = pd.DataFrame(
    scaler_cat8.fit_transform(X_train_m),
    columns=feature_cols
)
X_val_m_scaled = pd.DataFrame(
    scaler_cat8.transform(X_val_m),
    columns=feature_cols
)
X_test_m_scaled = pd.DataFrame(
    scaler_cat8.transform(X_test_m),
    columns=feature_cols
)

print("Scaling complete.")

Scaling complete.


In [6]:
# 5. Save artifacts

save_dir = '../extracted'
os.makedirs(save_dir, exist_ok=True)

# Binary branch
X_train_b_scaled.to_csv(f'{save_dir}/X_train_binary.csv', index=False)
X_val_b_scaled.to_csv(f'{save_dir}/X_val_binary.csv', index=False)
X_test_b_scaled.to_csv(f'{save_dir}/X_test_binary.csv', index=False)

y_train_b.to_csv(f'{save_dir}/y_train_binary.csv', index=False)
y_val_b.to_csv(f'{save_dir}/y_val_binary.csv', index=False)
y_test_b.to_csv(f'{save_dir}/y_test_binary.csv', index=False)

# Cat8 branch
X_train_m_scaled.to_csv(f'{save_dir}/X_train_cat8.csv', index=False)
X_val_m_scaled.to_csv(f'{save_dir}/X_val_cat8.csv', index=False)
X_test_m_scaled.to_csv(f'{save_dir}/X_test_cat8.csv', index=False)

y_train_cat8.to_csv(f'{save_dir}/y_train_cat8.csv', index=False)
y_val_cat8.to_csv(f'{save_dir}/y_val_cat8.csv', index=False)
y_test_cat8.to_csv(f'{save_dir}/y_test_cat8.csv', index=False)

# Fine34 labels aligned to cat8 split
y_train_fine34.to_csv(f'{save_dir}/y_train_fine34.csv', index=False)
y_val_fine34.to_csv(f'{save_dir}/y_val_fine34.csv', index=False)
y_test_fine34.to_csv(f'{save_dir}/y_test_fine34.csv', index=False)

# Feature list
with open(f'{save_dir}/feature_columns.json', 'w') as f:
    json.dump(feature_cols, f, indent=2)

# Encoders / mappings
with open(f'{save_dir}/fine34_mapping.json', 'w') as f:
    json.dump(fine34_mapping, f, indent=2)

with open(f'{save_dir}/cat8_mapping.json', 'w') as f:
    json.dump(CATEGORY_ENCODING, f, indent=2)

pickle.dump(fine_le, open(f'{save_dir}/label_encoder_fine34.pkl', 'wb'))
pickle.dump(scaler_binary, open(f'{save_dir}/scaler_binary.pkl', 'wb'))
pickle.dump(scaler_cat8, open(f'{save_dir}/scaler_cat8.pkl', 'wb'))

split_summary = {
    "binary_split": {
        "train_shape": list(X_train_b_scaled.shape),
        "val_shape": list(X_val_b_scaled.shape),
        "test_shape": list(X_test_b_scaled.shape),
        "train_distribution": y_train_b.value_counts().to_dict(),
        "val_distribution": y_val_b.value_counts().to_dict(),
        "test_distribution": y_test_b.value_counts().to_dict()
    },
    "cat8_split": {
        "train_shape": list(X_train_m_scaled.shape),
        "val_shape": list(X_val_m_scaled.shape),
        "test_shape": list(X_test_m_scaled.shape),
        "train_distribution": y_train_cat8.value_counts().sort_index().to_dict(),
        "val_distribution": y_val_cat8.value_counts().sort_index().to_dict(),
        "test_distribution": y_test_cat8.value_counts().sort_index().to_dict()
    }
}

with open(f'{save_dir}/split_summary.json', 'w') as f:
    json.dump(split_summary, f, indent=2)

print("Saved successfully.")

Saved successfully.
